# TCGA-BRCA Baseline Feature-Set V1 Review

This notebook is review-only. It reads saved baseline feature-set v1 outputs from disk, regenerates review tables in `05-results`, and does not parse raw files, freeze the endpoint, add treatment detail, or perform modeling.


## Load the latest saved baseline feature-set v1 run and regenerate review tables


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / '.git').exists():
            return candidate
    raise FileNotFoundError('Could not locate the repository root from the current working directory.')


def read_tsv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, sep='\t', dtype=str, keep_default_na=False)


def require_json_list(value: str, *, label: str) -> list[dict[str, str]] | list[str]:
    parsed = json.loads(value)
    if not isinstance(parsed, list):
        raise ValueError(f'Expected {label} to contain a JSON list.')
    return parsed


repo_root = find_repo_root(Path.cwd())
latest_pointer_path = (
    repo_root
    / '01-data'
    / 'audit'
    / 'tcga-brca'
    / 'analysis-prep'
    / 'tcga_brca_baseline_feature_set_v1_latest.json'
)
if not latest_pointer_path.exists():
    raise FileNotFoundError(
        f'Latest baseline feature-set pointer not found: {latest_pointer_path}. Run the feature-set script first.'
    )

latest_pointer = json.loads(latest_pointer_path.read_text(encoding='utf-8'))
feature_path = repo_root / latest_pointer['baseline_feature_set_v1_tsv']
audit_map_path = repo_root / latest_pointer['baseline_feature_set_v1_audit_map_tsv']
spec_path = repo_root / latest_pointer['baseline_feature_set_v1_spec_tsv']
missingness_path = repo_root / latest_pointer['baseline_feature_set_v1_missingness_tsv']
summary_path = repo_root / latest_pointer['baseline_feature_set_v1_summary_tsv']
run_log_path = repo_root / latest_pointer['run_log_json']
results_root = repo_root / '09-trials' / '01-tcga-only-source-audited' / '05-results'
results_root.mkdir(parents=True, exist_ok=True)

feature_df = read_tsv(feature_path)
audit_map_df = read_tsv(audit_map_path)
spec_df = read_tsv(spec_path)
missingness_df = read_tsv(missingness_path)
summary_df = read_tsv(summary_path)
run_log = json.loads(run_log_path.read_text(encoding='utf-8'))

if run_log.get('status') != 'completed':
    raise ValueError(f"Baseline feature-set run is not completed: status={run_log.get('status')}")
if not run_log.get('validation', {}).get('passed', False):
    raise ValueError('Baseline feature-set validation did not pass. Review the saved run_log.json before continuing.')
if len(feature_df) != len(audit_map_df):
    raise ValueError('baseline_feature_set_v1.tsv row count did not match baseline_feature_set_v1_audit_map.tsv.')

feature_with_row_index_df = (
    feature_df.reset_index(drop=True)
    .assign(feature_set_v1_row_index=lambda df: (df.index + 1).astype(str))
)
preview_df = (
    audit_map_df.merge(
        feature_with_row_index_df,
        on='feature_set_v1_row_index',
        how='inner',
        validate='one_to_one',
    )
    .sort_values(
        'feature_set_v1_row_index',
        key=lambda series: pd.to_numeric(series, errors='raise'),
    )
    .head(100)
    .reset_index(drop=True)
)

decision_order = {
    'include_in_feature_set_v1': 0,
    'exclude_from_feature_set_v1': 1,
}
spec_review_df = (
    spec_df.assign(
        feature_set_v1_decision_order=spec_df['feature_set_v1_decision'].map(decision_order).fillna(9)
    )
    .sort_values(
        ['feature_set_v1_decision_order', 'decision_rule', 'field_category', 'field_name'],
        ascending=[True, True, True, True],
    )
    .drop(columns=['feature_set_v1_decision_order'])
    .reset_index(drop=True)
)
missingness_review_df = (
    missingness_df.assign(
        missing_like_fraction_numeric=pd.to_numeric(missingness_df['missing_like_fraction'], errors='raise')
    )
    .sort_values(['missing_like_fraction_numeric', 'field_name'], ascending=[False, True])
    .drop(columns=['missing_like_fraction_numeric'])
    .reset_index(drop=True)
)
summary_review_df = summary_df.sort_values(['summary_section', 'summary_metric']).reset_index(drop=True)

review_outcomes_row = summary_review_df.loc[
    summary_review_df['summary_metric'] == 'review_field_decision_outcomes_json'
]
if len(review_outcomes_row) != 1:
    raise ValueError('Expected exactly one review_field_decision_outcomes_json summary row.')
review_decision_summary_df = pd.DataFrame(
    require_json_list(
        review_outcomes_row.iloc[0]['summary_value'],
        label='review_field_decision_outcomes_json',
    )
)
if review_decision_summary_df.empty:
    raise ValueError('Saved review_field_decision_outcomes_json summary row produced no review-decision rows.')
review_field_order = {
    'ethnicity': 0,
    'gender': 1,
    'icd_10': 2,
    'icd_o_3_site': 3,
}
review_decision_summary_df = (
    review_decision_summary_df.assign(
        review_field_order=review_decision_summary_df['field_name'].map(review_field_order).fillna(9)
    )
    .sort_values(['review_field_order', 'field_name'], ascending=[True, True])
    .drop(columns=['review_field_order'])
    .reset_index(drop=True)
)

included_feature_list_df = spec_review_df.loc[
    spec_review_df['feature_set_v1_decision'] == 'include_in_feature_set_v1',
    ['field_name', 'source_origin', 'candidate_bucket', 'decision_rule'],
].reset_index(drop=True)
readiness_review_df = summary_review_df.loc[
    summary_review_df['summary_section'] == 'readiness'
].reset_index(drop=True)

preview_path = results_root / '86_baseline_feature_set_v1_preview.tsv'
spec_review_path = results_root / '87_baseline_feature_set_v1_spec.tsv'
missingness_review_path = results_root / '88_baseline_feature_set_v1_missingness.tsv'
decision_summary_path = results_root / '89_baseline_feature_set_v1_decision_summary.tsv'
summary_review_path = results_root / '90_baseline_feature_set_v1_summary.tsv'

preview_df.to_csv(preview_path, sep='\t', index=False)
spec_review_df.to_csv(spec_review_path, sep='\t', index=False)
missingness_review_df.to_csv(missingness_review_path, sep='\t', index=False)
review_decision_summary_df.to_csv(decision_summary_path, sep='\t', index=False)
summary_review_df.to_csv(summary_review_path, sep='\t', index=False)

print(f"Baseline feature-set run ID: {latest_pointer['baseline_feature_set_v1_run_id']}")
print(f"Baseline profile run ID: {latest_pointer['baseline_profile_v1_run_id']}")
print(f"Baseline analysis run ID: {latest_pointer['baseline_analysis_v1_run_id']}")
print(f"Run log: {run_log_path}")
print(f"Saved: {preview_path}")
print(f"Saved: {spec_review_path}")
print(f"Saved: {missingness_review_path}")
print(f"Saved: {decision_summary_path}")
print(f"Saved: {summary_review_path}")

display(pd.DataFrame([latest_pointer]))
display(pd.DataFrame([run_log.get('validation', {})]))
display(included_feature_list_df)
display(missingness_review_df.head(20))
display(review_decision_summary_df)
display(readiness_review_df)
display(summary_review_df)


This notebook remains review-only. It must not be used to parse new raw inputs, perform imputation, collapse endpoint candidates into a final endpoint, reintroduce treatment detail, or train models.
